# Agent Skills: teaching an agent *how*, not just *what*

**Task (same as notebooks 01 & 02):** For profile `P001`, which genes connected
to autism spectrum disorder should we prioritize, and why?

---

## 0. From tool-calling to coding agents — why the paradigm shifted

**Where we've been.** In notebook 01 we handed the model a fixed list of typed
functions (`@tool`). In notebook 02 we served those same functions over MCP. In
both, the mental model is the same: a brain wired to a **hand-curated set of
tools**. A human designs every tool in advance, and the agent can only do what
those tools allow.

**The shift.** The frontier of agent design moved away from "bolt N bespoke
tools onto a model" toward "give the model a few **general** tools — *read a
file, write a file, run a shell command* — and let it work inside a real
**repository / filesystem**." Once an agent can run `bash` and edit files in a
working directory, it can install a package, run a script, inspect data, call a
CLI — without anyone pre-building a tool for each. The **interface becomes the
filesystem + shell**, not a fixed tool list. This is the *coding-agent*
paradigm: Claude Code, OpenAI Codex, Cursor, OpenHands, and friends.

**Why now / why this trend?**

- **Models got good enough** at long-horizon reasoning and writing correct code
  to be trusted with general-purpose tools instead of guard-railed narrow ones.
- **A few primitives generalize better than many bespoke tools.** `bash` + file
  I/O can express almost anything; a catalogue of 50 narrow tools is brittle and
  expensive to maintain.
- **The repo is shared memory.** Files persist across steps, the agent reads its
  own outputs, and `git` tracks what changed — the working directory becomes the
  agent's state, not just a place to dump results.

## 1. The new problem: how do you give such an agent *know-how*?

A coding agent with `bash` + files **can** do almost anything — but it doesn't
**know** your specifics: your team's procedure for ranking candidate genes, your
house answer format, the safety rules for biomedical claims. You *could* cram
all that into the system prompt, but then you pay for those tokens **every
turn**, even when the task is unrelated, and the prompt becomes an unmanageable
junk drawer as you add more.

### Agent Skills & `SKILL.md`

**Agent Skills** are the answer that emerged: package procedural knowledge into
a portable, version-controlled folder the agent loads **on demand**.

A skill is just a directory with a `SKILL.md` file:

```
my-skill/
├── SKILL.md       # required: YAML frontmatter (name, description) + instructions
├── scripts/       # optional: executable code the agent can run
├── references/    # optional: docs the agent can open when needed
└── assets/        # optional: templates, resources
```

The trick that makes this scale is **progressive disclosure** — the agent loads
skills in three stages, cheapest first:

1. **Discovery** — at startup the agent sees only each skill's `name` +
   `description` (a few tokens each). Just enough to know *when* it might apply.
2. **Activation** — when a task matches a skill's description, the agent reads
   the **full `SKILL.md`** into context.
3. **Execution** — it follows the instructions, optionally running bundled
   `scripts/` or opening `references/` files only if needed.

So you can keep *dozens* of skills on hand for almost no idle context cost — the
heavy content loads only when relevant. Agent Skills were created by Anthropic
and [released as an open standard](https://www.anthropic.com/engineering/equipping-agents-for-the-real-world-with-agent-skills)
(Dec 2025); within weeks Claude Code, Cursor, Codex, Gemini CLI, OpenHands,
Goose, Pi and others adopted the same `SKILL.md` format. Spec:
[agentskills.io](https://agentskills.io/specification).

### The harness

The model itself can't read a file, run `bash`, or do progressive disclosure —
it only emits text/tool-call requests. **Something has to** implement those
primitive tools, run the agent loop, manage the context window, and discover +
load skills at the right stage. That program is the **harness** (a.k.a. the
agent runtime / coding-agent).

> **Model : harness :: engine : car.** Claude Code, Codex, Cursor, OpenHands,
> and **Pi** are all harnesses. The model is the engine; the harness is
> everything around it that turns token prediction into a thing that edits your
> repo. Skills are a *format the harness knows how to find and load* — they are
> not part of the model.

## 2. How skills overlap with tools and MCP

Tools (nb01), MCP (nb02), and skills (nb03) are all ways to *extend* an agent —
but they sit at **different layers**, and they **compose** rather than compete:

| | What it adds | Form | Answers… |
|---|---|---|---|
| **Tools** (nb01) | a new **capability** | a typed function the model calls | *what can the agent do?* |
| **MCP** (nb02) | the same capabilities, **portably** | a standard server/transport for tools, resources, prompts | *where do the capabilities live, and who can reuse them?* |
| **Skills** (nb03) | **know-how / workflow** | a `SKILL.md` folder of instructions (+ optional scripts) | *how should the agent do it, and when?* |

The clean one-liner:

> **Tools & MCP = what the agent *can* do. Skills = what it *should* do, and how.**

**The edges are genuinely blurry, though** — that's worth being honest about:

- A skill can bundle `scripts/`, and the agent runs them via `bash`. Those
  scripts are, functionally, *tools* — just delivered inside a skill instead of
  registered up front.
- A skill's instructions can say "use the `toy_kg` MCP server for graph
  lookups." So a skill can *orchestrate* tools and MCP.
- Conversely, an MCP server also exposes **prompts** (nb02 §2) — templated
  instructions — which overlaps with what a skill provides.

So don't think of them as rival mechanisms. Think: **tools/MCP supply
capabilities; skills supply the procedural knowledge that decides which
capabilities to use, in what order, and how to report the result** — loaded only
when the task calls for it.

## 3. For this notebook: Pi

We'll drive the skill with **Pi**, a deliberately minimal terminal coding
harness. We picked it because it's lightweight and has first-class skills
support — see `planning/session-3-framework-plan.md` for the full option list
and rationale. Site: [pi.dev](https://pi.dev) · repo:
[github.com/earendil-works/pi](https://github.com/earendil-works/pi). Install
the CLI separately:

```bash
npm install -g --ignore-scripts @earendil-works/pi-coding-agent
# or: curl -fsSL https://pi.dev/install.sh | sh
```

> ⚠️ **Heads-up on shape.** Unlike nb01/nb02 (pure `import` + an in-process agent
> loop), Pi is a *terminal harness* — so this notebook drives it differently
> (via the shell / `subprocess`) rather than as a Python library. This is the
> one notebook that won't look like the other two, by nature of what a harness
> is. **Low-commitment:** if Pi proves too niche for participants, the same
> `skills/biomedical-kg-agent.SKILL.md` works with any skills-compatible harness
> (Claude Code, Goose, …) or even the plain "skill-as-system-prompt" approach —
> the skill file doesn't change.

## 4. Setup and the skill we'll hand to Pi

Bring-your-own-key from `.env` (same as nb01/nb02). We also locate the Pi CLI and
make sure Pi's `bash` tool uses this `eccb` interpreter, so it can
`import src.mofa_tools`.

In [1]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

# Bring-your-own-key, same as nb01/nb02 (model backend still undecided).
load_dotenv(PROJECT_ROOT / ".env")
assert os.environ.get("ANTHROPIC_API_KEY"), (
    "ANTHROPIC_API_KEY not found. Add it to a .env file in the project root."
)

SKILL_PATH = PROJECT_ROOT / "skills" / "biomedical-kg-agent.SKILL.md"
TASK = (
    "For profile P001, which genes connected to autism spectrum disorder "
    "should we prioritize, and why?"
)
MODEL = "anthropic/claude-sonnet-4-6"

# Pi is installed separately as a CLI (not a pip package):
#   npm install -g --ignore-scripts @earendil-works/pi-coding-agent
# (see https://pi.dev). Locate the binary.
PI = shutil.which("pi") or str(Path.home() / ".local" / "bin" / "pi")
assert Path(PI).exists(), f"Pi CLI not found at {PI}. Install it (see https://pi.dev)."

# Pi answers by running Python through its `bash` tool, so its shell must use an
# interpreter where `src.mofa_tools` imports — i.e. this same `eccb` env.
ENV_BIN = str(Path(sys.executable).parent)
PI_ENV = {**os.environ, "PATH": f"{ENV_BIN}:{Path(PI).parent}:{os.environ.get('PATH', '')}"}

print("pi binary :", PI)
print("skill     :", SKILL_PATH.relative_to(PROJECT_ROOT))
print("python    :", sys.executable)

pi binary : /Users/chaeeunlee/.local/bin/pi
skill     : skills/biomedical-kg-agent.SKILL.md
python    : /Users/chaeeunlee/anaconda3/envs/eccb/bin/python


The skill is a valid Agent Skill: YAML frontmatter (`name`, `description`) then
markdown instructions. It contains **no model-specific code** — just procedural
knowledge: when to use it, evidence discipline, the fixed answer format, the
biomedical-caution rules, and a short "how to query the graph in this repo"
section pointing at `src/mofa_tools.py`.

In [2]:
print(SKILL_PATH.read_text())

---
name: biomedical-kg-agent
description: Answer questions about biomedical knowledge graphs, multi-omic molecular profiles, and candidate gene prioritization. Use when a task involves graph nodes/edges, phenotype-gene associations, profile feature scores, or ranking candidate genes. Enforces tool-grounded evidence, a fixed answer format, and biomedical caution.
---

# Biomedical Knowledge Graph Agent Skill

## Purpose

Use this skill when answering questions about biomedical knowledge graphs, multi-omic molecular profiles, and candidate gene prioritization.

## Behavior

*   Use graph and profile tools before making factual claims about nodes, edges, paths, profile scores, or candidate rankings.
*   Prefer exact node identifiers over free-text names.
*   If a requested entity cannot be found in the graph, say so clearly and suggest the closest matching nodes when available.
*   Distinguish graph evidence from model interpretation.
*   Do not present biomedical associations as clinica

## 5. Run Pi as a real harness — *with* the skill

We drive Pi from the shell via `subprocess`. This is the key difference from
nb01/nb02: we are not importing a library and writing an agent loop — we hand a
**prompt to a coding agent** and let *it* run the loop, decide which of its
`read`/`bash`/`edit`/`write` tools to use, and load the skill.

The command is essentially:

```bash
pi -p "<task>" --skill skills/biomedical-kg-agent.SKILL.md \
   --model anthropic/claude-sonnet-4-6 --approve
```

- `-p` runs headless (process the prompt and exit) instead of opening the TUI.
- `--skill <path>` loads our Agent Skill (Pi also auto-discovers skills in
  `.pi/skills/`, `.agents/skills/`, …; we pass it explicitly here).
- `--approve` trusts project-local files for this run. Pi runs in
  `PROJECT_ROOT`, so its `bash` tool can run `python -c "from src.mofa_tools
  import ..."` directly.

Watch the output follow the skill's **Answer / Evidence Used / Interpretation /
Limitations** structure — that format comes from the skill, not from us.

In [3]:
def run_pi(prompt: str, with_skill: bool, model: str = MODEL, timeout: int = 300):
    """Invoke the Pi coding agent headlessly and return its final text answer."""
    cmd = [PI, "-p", prompt, "--model", model, "--approve"]
    cmd += ["--skill", str(SKILL_PATH)] if with_skill else ["--no-skills"]
    proc = subprocess.run(
        cmd,
        cwd=PROJECT_ROOT,
        env=PI_ENV,
        capture_output=True,
        text=True,
        timeout=timeout,
    )
    if proc.returncode != 0:
        print("[pi stderr]\n", proc.stderr[-2000:])
    return proc.stdout.strip()


with_skill_answer = run_pi(TASK, with_skill=True)
print(with_skill_answer)

Here are the full findings, formatted per the skill's answer structure:

---

## Answer

For profile **P001**, three genes are connected to Autism Spectrum Disorder (`HP:0000729`). They should be prioritized in this order:

| Rank | Gene | Profile Score | Pathway |
|------|------|--------------|---------|
| 1 | **SHANK3** | 2.4 | Synaptic signalling (`PATHWAY:SYNAPSE`) |
| 2 | **SCN2A** | 1.7 | Synaptic signalling (`PATHWAY:SYNAPSE`) |
| 3 | **CHD8** | 0.8 | Chromatin regulation (`PATHWAY:CHROMATIN`) |

---

## Evidence Used

- **Graph search:** `search_nodes(g, 'autism')` → identified `HP:0000729` (Autism spectrum disorder)
- **Candidate ranking:** `rank_candidate_genes(g, 'P001', 'HP:0000729')` → returned all three genes with their profile scores
- **Profile features:** `rank_profile_features('P001', top_n=10)` → confirmed the same scores: SHANK3 (2.4), SCN2A (1.7), CHD8 (0.8)
- **Graph edges:** All three genes carry an `associated_with` edge to `HP:0000729` (evidence: toy literature

## 6. Run *without* the skill — compare behaviour

Same agent, same model, same task — but with `--no-skills`. Pi can still *find*
`src/mofa_tools.py` by exploring the repo (it's a capable coding agent), so the
**capability** is unchanged. What changes is **behaviour**: without the skill it
has no instruction to ground every claim in tool output, to use the fixed
Answer/Evidence/Interpretation/Limitations format, or to add biomedical
caveats. Compare the two answers.

In [4]:
without_skill_answer = run_pi(TASK, with_skill=False)
print(without_skill_answer)

Here is the full, evidence-grounded answer:

---

## Gene Prioritization for Profile P001 — Autism Spectrum Disorder (HP:0000729)

The knowledge graph connects **three genes** to Autism Spectrum Disorder (ASD): *SHANK3*, *SCN2A*, and *CHD8*. Combining that graph structure with profile P001's multi-omic scores yields the following ranked list:

---

### 🥇 1. SHANK3 — **Top Priority** (profile score: **2.4**)

| Evidence type | Detail |
|---|---|
| **KG association** | `GENE:SHANK3 → associated_with → HP:0000729` (literature-backed) |
| **Pathway context** | Participates in **Synaptic signalling** (`PATHWAY:SYNAPSE`) |
| **Profile score** | **2.4** — highest of all ASD-linked genes in P001 |

> **Why?** SHANK3 carries the strongest molecular signal in this profile by a clear margin (+0.7 over SCN2A). It is directly linked to ASD in the graph **and** is a core component of synaptic signalling — the most biologically proximal pathway to ASD aetiology. The combination of high omics signal a

## Reflection

- **Same backend, three interfaces.** nb01 called the graph functions as bound
  tools, nb02 served them over MCP, and here Pi reached them by running
  `python` through its `bash` tool — yet all three produced `SHANK3, SCN2A,
  CHD8`. What does that tell you about where "capability" actually lives?
- **What did the skill change?** Diff the §5 (with-skill) and §6 (no-skill)
  answers. Which differences are *format*, which are *evidence discipline*, and
  which are *caution*? None of those are new capabilities.
- **Progressive disclosure:** we passed the skill explicitly with `--skill`. In
  a real session Pi would load only the skill's `name`/`description` until the
  task matched. Why does that matter once you have 50 skills instead of 1?
- **Harness vs library:** this notebook shells out to a CLI instead of importing
  a package. What did we gain (a real agent, repo access, on-demand skills) and
  what did we give up (in-process control, parallel structure with nb01/02)?
- **Overlap:** the skill told a coding agent how to call `src/mofa_tools.py`.
  If those functions were instead exposed via the nb02 MCP server, how would the
  skill's "how to query" section change — and what would stay the same?